# STEP 5 · JTSI 설계 (저널리즘 전환 스트레스 지수)

이 프로젝트의 **핵심 무기**다. 여러 스트레스 신호를 하나의 숫자로 합친다.

**JTSI = 디지털 피로 + AI 리스크 인식 + 역할 실행 갭 + 번아웃**

**z-score 표준화**: 각 변수를 '평균 0, 표준편차 1'로 변환해
공평한 출발선에 세운다. z = (값 - 평균) / 표준편차.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUT_TABLES = ROOT / "outputs" / "tables"
OUT_FIGURES = ROOT / "outputs" / "figures"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

import pyreadstat, pandas as pd, numpy as np
from scipy.stats import zscore
import warnings; warnings.filterwarnings('ignore')

df25,_ = pyreadstat.read_sav(DATA_RAW / "journalist_2025.sav")

def build(df):
    d = df.copy()
    d['fatigue']=d['q5']
    d['ai_user']=(d['q38']==1)
    d['risk']=d[['q39_4','q39_5','q39_6']].mean(1)
    d['burnout']=d[['q21_3','q21_4']].mean(1)
    d['satis']=d['q19_1']
    d['role_imp']=d[[f'q2_{i}' for i in range(1,8)]].mean(1)
    d['role_perf']=d[[f'q3_{i}' for i in range(1,8)]].mean(1)
    d['role_gap']=d['role_imp']-d['role_perf']
    return d

d25 = build(df25)

In [2]:
# 4요소를 각각 z-score로 표준화한 뒤 합산
comp = ['fatigue','risk','role_gap','burnout']
dd = d25.dropna(subset=comp).copy()
for c in comp:
    dd[c+'_z'] = zscore(dd[c])        # 표준화
dd['JTSI'] = dd[[c+'_z' for c in comp]].sum(1)   # 합산

print(f"분석 표본: {len(dd)}명")
print(f"JTSI 범위: {dd['JTSI'].min():.2f} ~ {dd['JTSI'].max():.2f}")
print(f"JTSI 평균: {dd['JTSI'].mean():.3f}  (z합산이라 0 근처가 정상)")

분석 표본: 2020명
JTSI 범위: -8.50 ~ 8.70
JTSI 평균: -0.000  (z합산이라 0 근처가 정상)


### 해석 가이드
- JTSI > 0 : 평균보다 전환 스트레스가 **높은** 사람
- JTSI < 0 : 평균보다 **낮은** 사람
- 범위 -8.5 ~ +8.7 : 개인 간 편차가 크다는 뜻

In [3]:
# 결과를 저장 (다음 단계에서 쓴다)
dd.to_csv(DATA_PROCESSED / "jtsi_2025.csv", index=False, encoding='utf-8-sig')
print("저장: data/processed/jtsi_2025.csv")

저장: data/processed/jtsi_2025.csv
